# 준지도학습 (Self-training / Pseudo-labeling) — Digit Recognizer (PyTorch) — Colab

라벨이 **아주 적을 때**, 라벨 없는 데이터를 **의사라벨(pseudo-label)** 로 활용해 성능을 끌어올리는 기본 예제입니다.

- 핵심 흐름: 소량 라벨로 baseline 학습 → 미라벨 데이터에 **고신뢰 예측만 의사라벨** 부여 → 합쳐서 재학습 → 향상 비교.
- IOAI 2025 **Antique Painting Authentication** 과제(라벨 일부만, 미라벨 적극 활용 = 준지도)의 기반 기법입니다.
  (공식 해법은 SpectralClustering+SVC+pseudo-label을 사용 — 여기서는 가장 기본인 self-training 으로 연습)
- [Digit Recognizer](https://www.kaggle.com/c/digit-recognizer) 데이터를 사용하며 토큰이 **노트북에 내장**되어 바로 실행됩니다.

> ⚙️ GPU 권장. ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부 공유 금지.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Digit Recognizer 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 라벨 소량 + 미라벨 분리

학습 데이터의 **1%만 라벨이 있다고 가정**하고, 나머지는 라벨 없는 데이터로 둡니다. 별도 검증셋으로 정확도를 측정합니다.

In [ ]:
import numpy as np, pandas as pd, torch
from torch.utils.data import TensorDataset, DataLoader

df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
X = df.drop("label", axis=1).values.reshape(-1,1,28,28).astype("float32")/255.0
y = df["label"].values.astype("int64")
rng = np.random.RandomState(42); idx = rng.permutation(len(X))
val_idx = idx[:5000]                # 검증
pool = idx[5000:]
n_lab = int(len(pool) * 0.01)       # 라벨 1%
lab_idx, unlab_idx = pool[:n_lab], pool[n_lab:]

Xval, yval = torch.from_numpy(X[val_idx]), y[val_idx]
Xlab, ylab = X[lab_idx], y[lab_idx]
Xunlab = X[unlab_idx]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| labeled:", len(lab_idx), "| unlabeled:", len(unlab_idx), "| val:", len(val_idx))

## 3. 모델 & 학습 유틸

In [ ]:
import torch.nn as nn, copy

def make_model():
    return nn.Sequential(
        nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(64*7*7,128), nn.ReLU(), nn.Dropout(0.25), nn.Linear(128,10)
    ).to(device)

def train(model, Xtr, ytr, epochs=8):
    loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr.astype("int64"))),
                        batch_size=64, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    return model

@torch.no_grad()
def evaluate(model):
    model.eval(); pred = []
    for i in range(0, len(Xval), 512):
        pred.append(model(Xval[i:i+512].to(device)).argmax(1).cpu().numpy())
    return (np.concatenate(pred) == yval).mean()

## 4. Baseline — 소량 라벨로만 학습

In [ ]:
base = train(make_model(), Xlab, ylab)
base_acc = evaluate(base)
print(f"[Baseline] 라벨 {len(ylab)}개로만 학습 → val acc {base_acc:.4f}")

## 5. Self-training — 미라벨에 의사라벨 부여 후 재학습

현재 모델로 미라벨 데이터를 예측해 **확신도(softmax 최대확률)가 높은 것만** 의사라벨로 채택하고, 라벨 데이터와 합쳐 재학습합니다.

In [ ]:
@torch.no_grad()
def predict_proba(model, arr, bs=512):
    model.eval(); out = []
    for i in range(0, len(arr), bs):
        p = torch.softmax(model(torch.from_numpy(arr[i:i+bs]).to(device)), dim=1)
        out.append(p.cpu().numpy())
    return np.concatenate(out)

model = base
CONF = 0.95
for rnd in range(1, 3):  # self-training 2 라운드
    proba = predict_proba(model, Xunlab)
    conf = proba.max(1); pseudo = proba.argmax(1)
    keep = conf >= CONF
    X_aug = np.concatenate([Xlab, Xunlab[keep]])
    y_aug = np.concatenate([ylab, pseudo[keep]])
    model = train(make_model(), X_aug, y_aug)
    acc = evaluate(model)
    print(f"[round {rnd}] 의사라벨 채택 {keep.sum()}/{len(keep)} (conf>={CONF}) → val acc {acc:.4f}")

print(f"\n=== Baseline {base_acc:.4f} → Self-training {acc:.4f} (향상 {acc-base_acc:+.4f}) ===")

## 🏁 제출 안내

이 노트북은 **준지도학습** 연습용 데모입니다(제출 리더보드 없음).

- 데이터 출처: **[Digit Recognizer](https://www.kaggle.com/c/digit-recognizer)**

> IOAI 2025 *Antique Painting Authentication*(라벨 일부 + 미라벨 활용) 과제의 기반 기법입니다. 공식 해법은 SpectralClustering+SVC+의사라벨을 결합합니다.